In [3]:
from collections import OrderedDict
from statistics import mean, pstdev
from typing import Dict, List, Set, Tuple

# ===========================
# Recall@k
# ===========================
def recall_at_k_single(gold_set: Set[str], ranked_ids: List[str], k: int) -> float:
    """
    單題 Recall@k = Top-k 中命中 gold_set 的「正解文物ID」數量 / gold_set 大小
    這裡 gold_set 裡放的是「文物名稱」，ranked_ids 也是文物名稱。
    """
    if not gold_set:
        return 0.0
    topk = ranked_ids[:k]
    hits = sum(1 for x in topk if x in gold_set)
    return hits / len(gold_set)


def evaluate_recall_for_param(
    ground_truth: Dict[str, Set[str]],
    runs_for_param: Dict[str, List[str]],
    k_list: List[int],
) -> Dict[int, Dict[str, float]]:
    """
    評估「單一參數組合」在多個 k 下的 Recall。
    - ground_truth: {qid -> set(正解文物名稱)}
    - runs_for_param: {qid -> [依序排名的文物名稱,...]}
    回傳:
      { k: { 'recall_mean': macro平均, 'recall_std': 題目間std, 'n_questions':題數,
             'recall_micro': micro加權平均 } }
    """
    qids = sorted(set(ground_truth.keys()) & set(runs_for_param.keys()))
    out: Dict[int, Dict[str, float]] = {}

    if not qids:
        for k in k_list:
            out[k] = {
                "recall_mean": 0.0,
                "recall_std": 0.0,
                "recall_micro": 0.0,
                "n_questions": 0,
            }
        return out

    for k in k_list:
        per_q: List[float] = []
        weights: List[int] = []
        for q in qids:
            r = recall_at_k_single(ground_truth[q], runs_for_param[q], k)
            per_q.append(r)
            weights.append(len(ground_truth[q]))

        macro = mean(per_q)
        std = pstdev(per_q) if len(per_q) > 1 else 0.0
        total_w = sum(weights)
        micro = (
            sum(r * w for r, w in zip(per_q, weights)) / total_w
            if total_w > 0
            else 0.0
        )

        out[k] = {
            "recall_mean": macro,
            "recall_std": std,
            "recall_micro": micro,
            "n_questions": len(per_q),
        }

    return out


def evaluate_param_grid(
    ground_truth: Dict[str, Set[str]],
    runs: Dict[str, Dict[str, List[str]]],
    k_list: List[int],
) -> Dict[str, Dict[int, Dict[str, float]]]:
    """
    對多個參數組合一起算 Recall。
    - runs: {param_key -> {qid -> [ranked_ids...]}}
    """
    results: Dict[str, Dict[int, Dict[str, float]]] = OrderedDict()
    for param_key, runs_for_param in runs.items():
        results[param_key] = evaluate_recall_for_param(
            ground_truth=ground_truth,
            runs_for_param=runs_for_param,
            k_list=k_list,
        )
    return results


def pick_best_by_k(
    summary: Dict[str, Dict[int, Dict[str, float]]],
    k: int = 10,
) -> Tuple[str, Dict[str, float]]:
    """
    在指定 k 下，選 recall_mean 最高者；若同分則選 recall_std 較小者。
    """
    candidates = []
    for param_key, metrics_by_k in summary.items():
        if k not in metrics_by_k:
            continue
        m = metrics_by_k[k]
        candidates.append((param_key, m["recall_mean"], m["recall_std"]))

    if not candidates:
        return "", {
            "recall_mean": 0.0,
            "recall_std": 0.0,
            "recall_micro": 0.0,
            "n_questions": 0,
        }

    # 先比 recall_mean，再比 -std（std 越小越好）
    candidates.sort(key=lambda x: (x[1], -x[2]), reverse=True)
    best_key, _, _ = candidates[0]
    return best_key, summary[best_key][k]


# ===========================
# 你的實際資料（5題 × 5參數）
# ===========================
# 正解（以文物名稱為ID）
ground_truth: Dict[str, Set[str]] = {
    "Q1": {"白瓷劃花蓮紋梅瓶"},
    "Q2": {"玉佛手"},
    "Q3": {"金碗"},
    "Q4": {"蒔繪菊籬螺鈿三層屜盒"},
    "Q5": {"白瓷劃花番蓮紋杯"},
}

# 各參數組合的 Top-10 排名結果
# param_key 命名你可以自己改，這裡是 size{chunk}_{overlap}
runs: Dict[str, Dict[str, List[str]]] = {
    "size1000_500": {
        # Q1
        "Q1": [
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "銀鍍金鏤空芙蓉花指甲套",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "瑪瑙花式碗",
            "黑地灑金星玻璃鼻煙壺",
            "宜興胎畫琺瑯海棠式茶壺",
            "宜興胎畫琺瑯提梁壺",
            "汝窯 青瓷洗 「奉華」銘",
            "鎏金鏤空纍絲竹葉紋指甲套",
        ],
        # Q2
        "Q2": [
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "玉鎖形佩",
            "玉佩",
            "銀鍍金鏤空芙蓉花指甲套",
            "青瓷觚",
            "象牙雕花卉荷包",
            "鎏金鏤空纍絲竹葉紋指甲套",
            "畫琺瑯西洋人物懷錶",
            "汝窯 青瓷洗 「奉華」銘",
        ],
        # Q3
        "Q3": [
            "金碗",
            "霽青釉描金蓋碗",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "瑪瑙花式碗",
            "銅鍍金嵌瑪瑙修妝匣",
            "黑地灑金星玻璃鼻煙壺",
            "金筷、匙",
            "宜興胎畫琺瑯提梁壺",
            "銀嵌金星玻璃鼻煙盒",
        ],
        # Q4
        "Q4": [
            "銅鍍金嵌瑪瑙修妝匣",
            "蒔繪菊籬螺鈿三層屜盒",
            "紅釉盤盛木雕彩繪佛手",
            "剔紅種菊圖捧盒",
            "汝窯 青瓷洗 「奉華」銘",
            "黑地灑金星玻璃鼻煙壺",
            "霽青釉描金蓋碗",
            "瑪瑙花式碗",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯提梁壺",
        ],
        # Q5
        "Q5": [
            "瓷胎仿剔紅蓋碗",
            "霽青釉描金蓋碗",
            "白瓷劃花番蓮紋杯",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "黑地灑金星玻璃鼻煙壺",
            "紅釉盤盛木雕彩繪佛手",
            "宜興胎畫琺瑯提梁壺",
            "白瓷水盂",
            "宜興胎畫琺瑯海棠式茶壺",
            "汝窯 青瓷洗 「奉華」銘",
        ],
    },

    "size600_300": {
        # Q1
        "Q1": [
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "瑪瑙花式碗",
            "銀鍍金鏤空芙蓉花指甲套",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "瑪瑙花式碗",
            "瑪瑙花式碗",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "黑地灑金星玻璃鼻煙壺",
        ],
        # Q2
        "Q2": [
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "玉鎖形佩",
            "玉佩",
            "畫琺瑯西洋人物懷錶",
            "銀鍍金鏤空芙蓉花指甲套",
            "玉鎖形佩",
            "青瓷觚",
            "汝窯 青瓷洗 「奉華」銘",
        ],
        # Q3
        "Q3": [
            "金碗",
            "霽青釉描金蓋碗",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "瑪瑙花式碗",
            "銅鍍金嵌瑪瑙修妝匣",
            "黑地灑金星玻璃鼻煙壺",
            "瓷胎仿剔紅蓋碗",
            "銀嵌金星玻璃鼻煙盒",
            "金筷、匙",
        ],
        # Q4
        "Q4": [
            "銅鍍金嵌瑪瑙修妝匣",
            "黑地灑金星玻璃鼻煙壺",
            "蒔繪菊籬螺鈿三層屜盒",
            "紅釉盤盛木雕彩繪佛手",
            "汝窯 青瓷洗 「奉華」銘",
            "剔紅種菊圖捧盒",
            "瑪瑙花式碗",
            "黑地灑金星玻璃鼻煙壺",
            "霽青釉描金蓋碗",
            "宜興胎畫琺瑯提梁壺",
        ],
        # Q5
        "Q5": [
            "瓷胎仿剔紅蓋碗",
            "霽青釉描金蓋碗",
            "白瓷劃花番蓮紋杯",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "汝窯 青瓷洗 「奉華」銘",
            "黑地灑金星玻璃鼻煙壺",
            "紅釉盤盛木雕彩繪佛手",
            "宜興胎畫琺瑯提梁壺",
            "白瓷水盂",
            "銅胎畫琺瑯黃地花卉紋套杯",
        ],
    },

    "size400_200": {
        # Q1
        "Q1": [
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "瑪瑙花式碗",
            "銀鍍金鏤空芙蓉花指甲套",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "宜興胎畫琺瑯提梁壺",
            "瑪瑙花式碗",
        ],
        # Q2
        "Q2": [
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "玉鎖形佩",
            "玉佩",
            "瑪瑙花式碗",
            "銀鍍金鏤空芙蓉花指甲套",
            "象牙雕花卉荷包",
        ],
        # Q3
        "Q3": [
            "金碗",
            "霽青釉描金蓋碗",
            "霽青釉描金蓋碗",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "銅鍍金嵌瑪瑙修妝匣",
            "宜興胎畫琺瑯提梁壺",
            "黑地灑金星玻璃鼻煙壺",
            "黑地灑金星玻璃鼻煙壺",
            "瑪瑙花式碗",
        ],
        # Q4
        "Q4": [
            "蒔繪菊籬螺鈿三層屜盒",
            "銅鍍金嵌瑪瑙修妝匣",
            "銅鍍金嵌瑪瑙修妝匣",
            "剔紅種菊圖捧盒",
            "霽青釉描金蓋碗",
            "宜興胎畫琺瑯提梁壺",
            "青瓷觚",
            "白瓷劃花蓮紋梅瓶",
            "汝窯 青瓷洗 「奉華」銘",
            "黑地灑金星玻璃鼻煙壺",
        ],
        # Q5
        "Q5": [
            "霽青釉描金蓋碗",
            "瓷胎仿剔紅蓋碗",
            "白瓷劃花番蓮紋杯",
            "宜興胎畫琺瑯提梁壺",
            "汝窯 青瓷洗 「奉華」銘",
            "黑地灑金星玻璃鼻煙壺",
            "宜興胎畫琺瑯海棠式茶壺",
            "白瓷水盂",
            "紅釉盤盛木雕彩繪佛手",
            "宜興胎畫琺瑯提梁壺",
        ],
    },

    "size200_100": {
        # Q1
        "Q1": [
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "瑪瑙花式碗",
            "瑪瑙扭絲紋碟",
            "黑地灑金星玻璃鼻煙壺",
        ],
        # Q2
        "Q2": [
            "紅釉盤盛木雕彩繪佛手",
            "玉佛手",
            "玉佛手",
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "象牙雕花卉荷包",
            "玉佩",
            "青瓷觚",
        ],
        # Q3
        "Q3": [
            "霽青釉描金蓋碗",
            "霽青釉描金蓋碗",
            "金碗",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "金鑲東珠貓睛石嬪妃朝冠",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "黑地灑金星玻璃鼻煙壺",
            "金鑲珊瑚菱花面簪",
            "瓷胎仿剔紅蓋碗",
        ],
        # Q4
        "Q4": [
            "蒔繪菊籬螺鈿三層屜盒",
            "銅鍍金嵌瑪瑙修妝匣",
            "黑地灑金星玻璃鼻煙壺",
            "剔紅種菊圖捧盒",
            "白瓷劃花蓮紋梅瓶",
            "蒔繪菊籬螺鈿三層屜盒",
            "宜興胎畫琺瑯提梁壺",
            "宜興胎畫琺瑯提梁壺",
            "瓷胎仿剔紅蓋碗",
            "銀嵌金星玻璃鼻煙盒",
        ],
        # Q5
        "Q5": [
            "瓷胎仿剔紅蓋碗",
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花番蓮紋杯",
            "白瓷劃花番蓮紋杯",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "宜興胎畫琺瑯提梁壺",
            "宜興胎畫琺瑯海棠式茶壺",
            "宜興胎畫琺瑯海棠式茶壺",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "白瓷水盂",
        ],
    },

    "size120_40": {
        # Q1
        "Q1": [
            "白瓷劃花蓮紋梅瓶",
            "白瓷劃花蓮紋梅瓶",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "白瓷劃花蓮紋梅瓶",
            "銅鍍金嵌瑪瑙修妝匣",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "白瓷劃花蓮紋梅瓶",
            "瑪瑙花式碗",
        ],
        # Q2
        "Q2": [
            "紅釉盤盛木雕彩繪佛手",
            "玉佛手",
            "玉佛手",
            "玉佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "紅釉盤盛木雕彩繪佛手",
            "玉佩",
            "玉佛手",
            "玉鎖形佩",
        ],
        # Q3
        "Q3": [
            "霽青釉描金蓋碗",
            "金碗",
            "霽青釉描金蓋碗",
            "瓷胎仿剔紅蓋碗",
            "瓷胎仿剔紅蓋碗",
            "金碗",
            "宜興胎畫琺瑯四季花卉蓋碗",
            "金鑲東珠貓睛石嬪妃朝冠",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "金碗",
        ],
        # Q4
        "Q4": [
            "蒔繪菊籬螺鈿三層屜盒",
            "銅鍍金嵌瑪瑙修妝匣",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "剔紅種菊圖捧盒",
            "銅鍍金嵌瑪瑙修妝匣",
            "宜興胎畫琺瑯提梁壺",
            "宜興胎畫琺瑯提梁壺",
            "宜興胎畫琺瑯提梁壺",
            "瓷胎仿剔紅蓋碗",
            "青瓷觚",
        ],
        # Q5
        "Q5": [
            "白瓷劃花番蓮紋杯",
            "白瓷水盂",
            "瓷胎仿剔紅蓋碗",
            "銅胎畫琺瑯黃地花卉紋套杯",
            "白瓷劃花蓮紋梅瓶",
            "青瓷觚",
            "瓷胎仿剔紅蓋碗",
            "宜興胎畫琺瑯海棠式茶壺",
            "霽青釉描金蓋碗",
            "白瓷劃花蓮紋梅瓶",
        ],
    },
}

# ===========================
# 執行評估 & 印結果
# ===========================
if __name__ == "__main__":
    k_list = [1, 3, 5, 10]

    summary = evaluate_param_grid(
        ground_truth=ground_truth,
        runs=runs,
        k_list=k_list,
    )

    # 印出每個參數在各 k 的 Recall
    for param_key, metrics in summary.items():
        print(f"\n=== {param_key} ===")
        for k in k_list:
            m = metrics[k]
            print(
                f"k={k:<2} | "
                f"macro={m['recall_mean']:.3f} | "
                f"micro={m['recall_micro']:.3f} | "
                f"std={m['recall_std']:.3f} | "
                f"n={m['n_questions']}"
            )

    # 例如：在 k=5 挑出最佳參數
    best_key, best_m = pick_best_by_k(summary, k=5)
    print(
        f"\n>> Best @k=5: {best_key} | "
        f"macro={best_m['recall_mean']:.3f} | "
        f"std={best_m['recall_std']:.3f} | "
        f"n={best_m['n_questions']}"
    )



=== size1000_500 ===
k=1  | macro=0.600 | micro=0.600 | std=0.490 | n=5
k=3  | macro=1.000 | micro=1.000 | std=0.000 | n=5
k=5  | macro=1.000 | micro=1.000 | std=0.000 | n=5
k=10 | macro=1.000 | micro=1.000 | std=0.000 | n=5

=== size600_300 ===
k=1  | macro=0.600 | micro=0.600 | std=0.490 | n=5
k=3  | macro=1.200 | micro=1.200 | std=0.400 | n=5
k=5  | macro=1.200 | micro=1.200 | std=0.400 | n=5
k=10 | macro=1.200 | micro=1.200 | std=0.400 | n=5

=== size400_200 ===
k=1  | macro=0.800 | micro=0.800 | std=0.400 | n=5
k=3  | macro=1.400 | micro=1.400 | std=0.490 | n=5
k=5  | macro=1.600 | micro=1.600 | std=0.800 | n=5
k=10 | macro=1.600 | micro=1.600 | std=0.800 | n=5

=== size200_100 ===
k=1  | macro=0.400 | micro=0.400 | std=0.490 | n=5
k=3  | macro=1.600 | micro=1.600 | std=0.800 | n=5
k=5  | macro=2.000 | micro=2.000 | std=0.894 | n=5
k=10 | macro=2.600 | micro=2.600 | std=1.356 | n=5

=== size120_40 ===
k=1  | macro=0.600 | micro=0.600 | std=0.490 | n=5
k=3  | macro=1.400 | micro=1